# 01 · Border rivers at 10 m: does water actually cross the Bangladesh–India border?

Stage A1 of **bangladesh-water-ml**. For every modelled river crossing of the India-facing border
(HydroRIVERS channels ≥10 km² catchment × BBS 2023 boundary; 1,675 crossings), take the least-cloudy
Sentinel-2 L2A scene for a dry window (Jan–Mar) and a monsoon window (Aug–Oct), build an NDWI water mask
at 10 m, and test whether one connected water body touches the border within 500 m of the crossing
**and** has water on both sides of the line.

Data (all open): Sentinel-2 L2A via Microsoft Planetary Computer (no login) · BBS administrative
boundaries v03 via OCHA HDX · geoBoundaries IND/MMR ADM0 · crossing table in `data/crossings.csv` (this repo).

Runtime: CPU is enough (I/O bound). ~1,675 crossings × 2 seasons ≈ 25 min with 8 threads.

In [ ]:
!pip -q install pystac-client planetary-computer rasterio geopandas pyogrio scipy shapely folium

In [ ]:
import os, zipfile, urllib.request, json
os.makedirs('raw/bbs', exist_ok=True)
HDX = 'https://data.humdata.org/dataset/401d3fae-4262-48c9-891f-461fd776d49b/resource/05d85fe2-40c0-4b57-b1b5-be9c36a592fb/download/bgd_admin_boundaries.shp.zip'
if not os.path.exists('raw/bbs/bgd_admin0.shp'):
    urllib.request.urlretrieve(HDX, 'raw/bbs.zip'); zipfile.ZipFile('raw/bbs.zip').extractall('raw/bbs')
for iso, name in [('IND', 'ind'), ('MMR', 'mmr')]:
    p = f'raw/{name}_adm0.geojson'
    if not os.path.exists(p):
        meta = json.load(urllib.request.urlopen(f'https://www.geoboundaries.org/api/current/gbOpen/{iso}/ADM0/'))
        urllib.request.urlretrieve(meta['gjDownloadURL'], p)
CSV = 'crossings.csv'
if not os.path.exists(CSV):
    try:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/deluair/bangladesh-water-ml/main/data/crossings.csv', CSV)
    except Exception as e:
        print('Download failed, upload crossings.csv via the Files pane:', e)
print(sorted(os.listdir('raw')))

In [ ]:
"""A1: does a connected water body actually cross the Bangladesh-India border near each modelled crossing?

Per crossing and season (dry Jan-Mar, monsoon Aug-Oct; YEAR): least-cloudy Sentinel-2 L2A scene from
Microsoft Planetary Computer (CQL2 cloud filter), 1 km chip of B03/B08 at 10 m, NDWI > 0 water mask,
morphological clean-up, connected components. The India-facing border (BBS adminlines adm_level 0,
nearer India than Myanmar) is rasterised into the chip. A crossing is CONFIRMED in a season when a
water component touches the border within SEARCH_M of the modelled point and has water pixels on both
sides of the line. Reports the border-crossing width (water pixels on the border line x 10 m).
"""
from __future__ import annotations

import sys
from concurrent.futures import ThreadPoolExecutor

import geopandas as gpd
import numpy as np
import pandas as pd
import planetary_computer as pc
import pystac_client
import rasterio
from rasterio.features import rasterize
from rasterio.windows import from_bounds
from scipy import ndimage
from shapely.geometry import Point, box
from shapely.ops import unary_union

YEAR = 2024
SEASONS = {"dry": (f"{YEAR}-01-01/{YEAR}-03-31", 10), "monsoon": (f"{YEAR}-08-01/{YEAR}-10-31", 40)}
HALF = 500  # metres
SEARCH_M = 500

cat = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)


def load_border(raw: str) -> gpd.GeoSeries:
    lines = gpd.read_file(f"{raw}/bbs/bgd_adminlines.shp").to_crs(32646)
    intl = lines[lines.adm_level == 0].copy()
    ind = unary_union(gpd.read_file(f"{raw}/ind_adm0.geojson").to_crs(32646).geometry)
    mmr = unary_union(gpd.read_file(f"{raw}/mmr_adm0.geojson").to_crs(32646).geometry)
    keep = intl.geometry.distance(ind) <= intl.geometry.distance(mmr)
    bd = unary_union(gpd.read_file(f"{raw}/bbs/bgd_admin0.shp").to_crs(32646).geometry)
    return gpd.GeoSeries([unary_union(intl[keep].geometry)], crs=32646), gpd.GeoSeries([bd], crs=32646)


def scene(lon: float, lat: float, window: str, cloud: int):
    items = list(cat.search(collections=["sentinel-2-l2a"], intersects={"type": "Point", "coordinates": [lon, lat]},
                            datetime=window, filter_lang="cql2-json",
                            filter={"op": "<", "args": [{"property": "eo:cloud_cover"}, cloud]}).items())
    return min(items, key=lambda i: i.properties["eo:cloud_cover"]) if items else None


def one(r, border, bd) -> dict:
    out = {"idx": r.Index}
    pt = gpd.GeoSeries([Point(r.lon, r.lat)], crs=4326)
    for season, (win, cloud) in SEASONS.items():
        try:
            it = scene(r.lon, r.lat, win, cloud)
            if it is None:
                out[f"{season}_status"] = "no_scene"
                continue
            with rasterio.open(it.assets["B03"].href) as g_src, rasterio.open(it.assets["B08"].href) as n_src:
                p = pt.to_crs(g_src.crs).iloc[0]
                win_ = from_bounds(p.x - HALF, p.y - HALF, p.x + HALF, p.y + HALF, g_src.transform)
                g = g_src.read(1, window=win_, boundless=True, fill_value=0).astype("float32")
                n = n_src.read(1, window=win_, boundless=True, fill_value=0).astype("float32")
                tr = g_src.window_transform(win_)
                crs = g_src.crs
            if (g == 0).mean() > 0.2:
                out[f"{season}_status"] = "nodata"
                continue
            water = ((g - n) / np.maximum(g + n, 1)) > 0
            water = ndimage.binary_opening(water, iterations=1)
            lab, _ = ndimage.label(water)
            chipgeom = box(p.x - HALF, p.y - HALF, p.x + HALF, p.y + HALF)
            b = border.to_crs(crs).iloc[0].intersection(chipgeom)
            if b.is_empty:
                out[f"{season}_status"] = "border_outside_chip"
                continue
            near = b.intersection(p.buffer(SEARCH_M))
            bline = rasterize([(near.buffer(10), 1)], out_shape=g.shape, transform=tr, fill=0).astype(bool)
            inside = rasterize([(bd.to_crs(crs).iloc[0], 1)], out_shape=g.shape, transform=tr, fill=0).astype(bool)
            best_w, confirmed = 0, False
            for k in np.unique(lab[bline & water]):
                comp = lab == k
                both = (comp & inside & ~bline).any() and (comp & ~inside & ~bline).any()
                width = int((comp & bline).sum() / 2)  # buffered line is ~2 px thick
                if both:
                    confirmed = True
                    best_w = max(best_w, width * 10)
            out[f"{season}_status"] = "confirmed" if confirmed else "no_water_crossing"
            out[f"{season}_width_m"] = best_w
            out[f"{season}_scene"] = it.id
            out[f"{season}_cloud"] = round(it.properties["eo:cloud_cover"], 1)
        except Exception as exc:  # keep going; record why
            out[f"{season}_status"] = f"error:{type(exc).__name__}"
    return out




In [ ]:
import pandas as pd
N = 40   # set 0 for all 1,675 crossings
c = pd.read_csv(CSV)
if N: c = c.sample(N, random_state=20260927)
border, bd = load_border('raw')
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(lambda r: one(r, border, bd), c.itertuples()))
out = c.join(pd.DataFrame(res).set_index('idx'))
out.to_csv('a1_results.csv')
for s in SEASONS: print(s, out[f'{s}_status'].value_counts().to_dict())
grp = pd.cut(out.gsw_max_occ, [-1, 9.99, 49.99, 100], labels=['<10', '10-49', '>=50'])
(out.assign(dry=out.dry_status.eq('confirmed'), monsoon=out.monsoon_status.eq('confirmed'))
    .groupby(grp, observed=True)[['dry', 'monsoon']].mean().round(3))

In [ ]:
import folium
m = folium.Map(location=[24.0, 90.3], zoom_start=7, tiles=None)
folium.TileLayer('https://tiles.maps.eox.at/wmts/1.0.0/s2cloudless_3857/default/g/{z}/{y}/{x}.jpg',
                 attr='Sentinel-2 cloudless 2016 by EOX (CC BY 4.0)').add_to(m)
col = {'confirmed': '#00e5ff', 'no_water_crossing': '#ffffff'}
for r in out.itertuples():
    s = r.monsoon_status if isinstance(r.monsoon_status, str) else 'na'
    folium.CircleMarker([r.lat, r.lon], radius=4, color=col.get(s, '#888'), fill=True,
                        popup=f"Landsat {r.gsw_max_occ}% · dry {r.dry_status} · monsoon {s} · width {getattr(r, 'monsoon_width_m', '')} m").add_to(m)
m